In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace=underlying_embeddings.model
)


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PDFPlumberLoader


loader = PDFPlumberLoader("./pdf-file.pdf")

docs = loader.load()


text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)

recursive_docs = text_splitter.split_documents(docs)

vectorstore = InMemoryVectorStore.from_documents(
    documents=recursive_docs,
    embedding=cached_embedder
)


In [4]:
from langchain_chroma import Chroma

# 1. DB 경로 설정
CHROMA_PATH = "./chroma_db"

# 2. Store 생성
db = Chroma.from_documents(
    documents=recursive_docs,
    embedding=cached_embedder,
    persist_directory=CHROMA_PATH,
    collection_name="rag_collection")

In [5]:
query = "Tesla 투자 비중이 얼마나 되나요?"
results = db.similarity_search(query, k=1)
# 벡터스토어에 있는 문서와 유사도를 계산하고, 유사도가 높은 순서대로 검색 결과를 반환한다

print(len(results))

print(f"검색된 문서 내용:\n{results[0].page_content}")
# 가장 유사한 문서의 내용을 반환한다
# 이 내용을 LLM에게 전달해서 답변을 생성할 수 있다
# 이것이 RAG 

1
검색된 문서 내용:
Holdings Data - ARKK
As of 11/26/2025
ARKK
ARK Innovation ETF
Company Ticker CUSIP Shares Market Value ($) Weight (%)
1 TESLA INC TSLA 88160R101 2,204,438 $924,541,297.20 12.26%
2 TEMPUS AI INC TEM 88023B103 5,465,331 $419,956,034.04 5.57%
3 ROKU INC ROKU 77543R102 4,347,025 $412,445,732.00 5.47%


**질문(Query)은 청킹(쪼개기)을 하지 않습니다.**

상세한 과정을 단계별로 짚어드릴게요.

### 1. 질문(Query) 처리 과정
`db.similarity_search(query, k=1)` 명령을 내리면 내부적으로 다음과 같은 일이 벌어집니다.

1.  **임베딩 (청킹 X):** 질문인 "Tesla 투자 비중이 얼마나 되나요?"라는 문장 **전체를 통째로** 숫자로 바꿉니다(임베딩). 질문은 보통 짧기 때문에 쪼갤 필요가 없습니다.
2.  **유사도 비교:** 질문의 숫자 덩어리(벡터)와 이미 `store`에 저장되어 있는 수많은 문서 조각(청크)들의 숫자 덩어리들을 수학적으로 비교합니다.
3.  **결과 추출 (k=1):** 비교 결과, 질문과 **가장 비슷하게 생긴 문서 조각 딱 1개**를 골라냅니다.

---

### 2. 왜 질문은 청킹하지 않나요?
*   **문서(Documents):** 내용이 너무 길어서 LLM이 한 번에 읽을 수 없으므로 잘게 쪼개서 저장합니다.
*   **질문(Query):** 질문 자체가 하나의 완성된 의도입니다. 만약 "Tesla 투자 비중이 / 얼마나 되나요?"라고 쪼개서 검색하면, 각각의 조각은 원래 질문의 의미를 잃어버릴 수 있습니다. 그래서 질문은 항상 **통째로** 임베딩합니다.

---

### 3. "자동으로" 일어나는 일들
사용자님이 `db.similarity_search` 한 줄만 썼을 때, LangChain이 배후에서 자동으로 해주는 일은 다음과 같습니다.
*   `query`를 임베딩 모델(OpenAI 등)로 보냄.
*   받아온 벡터를 벡터 스토어의 데이터와 비교함.
*   가장 유사한 순서대로 정렬해서 상위 `k`개를 반환함.

### 요약
- **질문:** 쪼개지 않고 **통째로** 임베딩합니다.
- **비교:** 질문 벡터 vs 저장된 청크 벡터들.
- **결과:** 유사도가 가장 높은 **상위 1개(k=1)**의 문서 조각을 가져옵니다.

이것이 바로 RAG의 첫 번째 단계인 **'검색(Retrieval)'** 과정입니다!